# Create embeddings of documents

#### Setup Environment

In [1]:
from os import getenv, makedirs 
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
from semantic_text_splitter import TextSplitter
from tokenizers import Tokenizer
import pickle

In [2]:
# Loads variables from the environment
load_dotenv()
open_api_key = getenv("OPENAI_API_KEY")

#### Read in and Chunk Documents 

#### Create Embedding Vectors for Chunks

> ToDo: add some sort of loop through all videos

In [3]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_key', 'title'])
title_df.head()


,video_key,title
0,4b6bwcWK6GE,Welcome to the Huberman Lab Podcast
1,H-XfCl-HpRM,How Your Brain Works & Changes
2,nm1TxQj9IsQ,Master Your Sleep & Be More Alert When Awake
3,nwSkFq4tyC0,"Using Science to Optimize Sleep, Learning & Me..."
4,NAATB55oxeQ,"How to Defeat Jet Lag, Shift Work & Sleeplessness"


In [4]:
client = OpenAI(api_key=open_api_key)

max_tokens = 1023 # 8191 is max length for text-embedding-3-large
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")
splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, max_tokens)

# Create lists to store the data
chunk_embeddings = []
title_embeddings = []
ids = []
video_keys = []
chunk_keys = []
chunk_dict = {}


# Iterates through a list of video IDs, where each ID corresponds to a podcast episode transcript.
# Each transcript will be split into chunks and embedded along with its title for semantic search.
documents = list(title_dict.keys())  # ToDo: process using full dataset 
# documents = ['4b6bwcWK6GE', 'H-XfCl-HpRM']
for idx, video_key in enumerate(documents):

    # Create Title Embeddings
    title_vector = client.embeddings.create(
        input=title_dict[video_key],
        model="text-embedding-3-small"
    )

    # Load document and split into semantic chunks
    try:
        with open(f'data/documents/{video_key}.txt', 'r', encoding='utf-8') as file:
            text_content = file.read()
    except FileNotFoundError:
        print(f"Warning: Document file not found for video ID: {video_key}")
        continue
    # ToDo: validate or rework so chunks are topical sentiments with varying lengths
    chunks = splitter.chunks(text_content) 

    # Create data record of embeddings for each chunk of the transcript
    for chunk_idx, chunk in enumerate(chunks):
        # Create chunks directory if it doesn't exist 
        makedirs('data/chunks', exist_ok=True)
        
        # Write chunk to file with video ID and chunk number
        # Use a more filesystem-friendly naming convention
        chunk_key = f'{video_key}_chunk_{chunk_idx}'
        chunk_filename = f'data/chunks/{chunk_key}.txt'
        
        # Write chunk directly to file
        with open(chunk_filename, 'w', encoding='utf-8') as f:
            f.write(chunk)
        
        # Add chunk to utility dictionary
        chunk_dict[chunk_key] = chunk

        # Create Chunk Embeddings
        chunk_vector = client.embeddings.create(
            input=chunk,
            model="text-embedding-3-small"
        )

        # Append values of this record to column list
        chunk_embeddings.append(chunk_vector.data[0].embedding)
        title_embeddings.append(title_vector.data[0].embedding)
        ids.append(idx)
        video_keys.append(video_key)
        chunk_keys.append(chunk_key)
print("Took 50min52sec to run locally on macbook 2015 pro")

/Users/dom/Library/Mobile Documents/com~apple~CloudDocs/Ohmic Data/Repositories/huberman-lab-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


50min52sec to run locally using 2015 Quad-Core Intel Core i7

In [5]:
with open('data/chunk_dict.pkl', 'wb') as f:
    pickle.dump(chunk_dict, f)

In [6]:
# Create DataFrame
df = pd.DataFrame({
    'id': ids,
    'video_key': video_keys, 
    'chunk_key': chunk_keys,
    'content_vector': chunk_embeddings,
    'title_vector': title_embeddings,
})
df.reset_index(inplace=True)
df.rename(columns={'index': 'vector_id'}, inplace=True)
df.head()

In [8]:
# Save DataFrame to CSV
df.to_csv('data/embeddings.csv', index=False)